# Covariate Balance and Matching: Video Level
---
**Context:** Corresponding to the algorithmic distribution comparison. Ensures that AIGC videos and HGC videos shared comparable pre-upload creator characteristics.

**Variables:**
- **Treatment (D):** `AIGC Video` — The uploaded video being AIGC (1 = AIGC, 0 = HGC).
- **Outcome (Y):** `Algorithmic Exposure Metrics` — e.g., 31-day Cumulative Exposure, Days to Reach 90% Exposure.

**Matching Objective:**
To accurately assess the structural impact of AIGC, this notebook performs exact and nearest-neighbor matching to mitigate selection bias. The procedure ensures that the treatment and control groups share highly comparable baseline characteristics prior to downstream estimation. SMD stands for Standardized Mean Difference. VR stands for Variance Ratio. Following Rubin's recommendations, covariates are considered adequately balanced if $|SMD| < 0.1$ and $VR \in [0.5, 2.0]$.

### Step 1: Environment Setup & Data Standardization
Import the core matching engine and prepare the dataset. Continuous covariates are standardized using Z-score normalization to ensure equal weighting in the Euclidean distance calculation.

In [ ]:
import sys
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Import core matching algorithm
sys.path.append('../core')
from matching import run_parallel_matching, evaluate_balance

# Load cleaned data
df = pd.read_parquet('../_data_preprocessing/clean_matching_video.parquet')
treatment_col = 'AIGC Video'
exact_cols = ['Creator Category', 'Video Upload Date', 'Brand Tier', 'Creator City']
numerical_cols = ['Creator Video Uploads (14 days)', 'Creator Video Uploads (60 days)', 'Creator Total Play Hours (14 days)', 'Creator Total Play Hours (60 days)', 'Creator Total Play Count (14 days)', 'Creator Total Play Count (60 days)', 'Creator Follower Count']

# Drop missing values and standardize numerical features
df.dropna(subset=exact_cols + numerical_cols + [treatment_col], inplace=True)
df[[f'scaled_{c}' for c in numerical_cols]] = StandardScaler().fit_transform(df[numerical_cols])


### Step 2: Exact Matching (Hash Blocking)
To enforce strict comparability on categorical dimensions (e.g., City, Category), we construct a unique hash key for each record. A treated unit is strictly blocked from matching with any control unit that does not share the exact same hash.

In [ ]:
# Assign hash keys for exact matching constraints
treated = df[df[treatment_col] == 1].assign(cate_hash=df[exact_cols].astype(str).apply('_'.join, axis=1))
control = df[df[treatment_col] == 0].assign(cate_hash=df[exact_cols].astype(str).apply('_'.join, axis=1))

def build_hash_map(data):
    m = {}
    for i, v in enumerate(data['cate_hash']): m.setdefault(v, []).append(i)
    return m

t_map, c_map = build_hash_map(treated), build_hash_map(control)
t_grp, c_grp = treated.reset_index(drop=True), control.reset_index(drop=True)


### Step 3: Nearest-Neighbor Matching Execution
Execute the GPU-accelerated multi-process matching algorithm. We apply a 1:1 Nearest Neighbor (NN) matching with replacement, subject to specific threshold calipers to prune poor matches.

In [ ]:
# Set distance threshold calipers
thresh = {'default': 0.7}

# Run parallel matching
idx_mp, dist_mp = run_parallel_matching(t_map, c_map, t_grp, c_grp, numerical_cols, thresh)

# Reconstruct matched dataset
t_grp['match_id'] = [c_grp.iloc[i]['ID'] if i != -1 else -1 for i in idx_mp]
p_treat = t_grp[t_grp['match_id'] != -1].copy()
m_ctrl = p_treat[['match_id']].merge(c_grp, left_on='match_id', right_on='ID', how='left')

print(f'Successfully matched pairs: {len(p_treat)}')


### Step 4: Covariate Balance Evaluation
Calculate Standardized Mean Difference (SMD) and Variance Ratio (VR) to verify that selection bias has been successfully mitigated. The output is formatted for direct inclusion in the manuscript's Supplementary Information.

In [ ]:
# Evaluate and output balance metrics
bal_df, counts = evaluate_balance(df, treatment_col, numerical_cols, p_treat, m_ctrl, 'video_results')
print('=== Covariate Balance Test ===')
display(bal_df.round(4))
